In [0]:
%pip install catboost openpyxl

In [0]:
from pathlib import Path
import pandas as pd

# Datenordner aus deinem aktuellen NMF-Notebook
basis = Path("/Workspace/Users/khalil-said.albert@de.abb.com/Daten")

# Passende Excel-Dateien suchen
bom_dateien = sorted(basis.glob("191007_Stuecklisten.aufgeloest (002)_korrigiert.xlsx"))

# Benötigte Dateien prüfen
datei_check = pd.DataFrame([
    {"Prüfung": "bom_core.py vorhanden",
     "OK": (basis / "bom_core.py").is_file()},
    {"Prüfung": "layer2_profile.py vorhanden",
     "OK": (basis / "layer2_profile.py").is_file()},
    {"Prüfung": "Genau eine korrigierte Stückliste vorhanden",
     "OK": len(bom_dateien) == 1},
])

display(datei_check)

In [0]:
import sys
import time

assert datei_check["OK"].all(), "Bitte zuerst die fehlenden Dateien prüfen."

# Python soll eure Module im Datenordner finden
if str(basis) not in sys.path:
    sys.path.insert(0, str(basis))

import bom_core as bc
import layer2_profile as L2

# Die zuvor gefundene Stückliste mit dem bestehenden Cleaner laden
arbeitsdatei = str(bom_dateien[0])
start = time.perf_counter()

sauber, protokoll = bc.clean_bom(arbeitsdatei)

display(pd.DataFrame([{
    "Positionszeilen": len(sauber),
    "Erwartete Positionszeilen": 47390,
    "Stand stimmt überein": len(sauber) == 47390,
    "Laufzeit in Sekunden": round(time.perf_counter() - start, 1)
}]))

In [0]:
# Aus der Liste eine Tabelle mit den drei benötigten Spalten bilden
positionen = pd.DataFrame(sauber)[
    ["erzeugnis", "block", "komponente"]
].copy()

assert positionen.notna().all().all(), "Produkt, Block oder Komponente fehlt."

# Pro Produktblock die verschiedenen Komponenten zusammenstellen
block_sets = (
    positionen.groupby(["erzeugnis", "block"])["komponente"]
    .agg(lambda komponenten: tuple(sorted(set(komponenten))))
    .reset_index(name="komponentenset")
)

# Pro Produkt die Blöcke und unterschiedlichen Komponentensets zählen
block_check = (
    block_sets.groupby("erzeugnis")
    .agg(
        bloecke=("block", "size"),
        unterschiedliche_sets=("komponentenset", "nunique")
    )
)

display(pd.DataFrame([{
    "Produkte": len(block_check),
    "Produktblöcke": len(block_sets),
    "Produkte mit mehreren Blöcken": int((block_check["bloecke"] > 1).sum()),
    "Davon mit unterschiedlichen Komponentensets":
        int((block_check["unterschiedliche_sets"] > 1).sum())
}]))

In [0]:
# Gleiche Komponentensets innerhalb desselben Produkts einmal zählen
fassungen = (
    block_sets[["erzeugnis", "komponentenset"]]
    .drop_duplicates(["erzeugnis", "komponentenset"])
    .reset_index(drop=True)
)

# Jede unterschiedliche Fassung bekommt eine technische Kennung
fassungen["fassung_id"] = fassungen.index

# Alle Fassungen eines Produkts erhalten zusammen das Gewicht 1
anzahl_fassungen = (
    fassungen.groupby("erzeugnis")["fassung_id"].transform("count")
)
fassungen["produktgewicht"] = 1.0 / anzahl_fassungen

display(pd.DataFrame([{
    "Produkte": fassungen["erzeugnis"].nunique(),
    "Unterschiedliche Fassungen": len(fassungen),
    "Produkte mit mehreren Fassungen":
        fassungen.loc[anzahl_fassungen > 1, "erzeugnis"].nunique(),
    "Summe der Produktgewichte": fassungen["produktgewicht"].sum()
}]))

In [0]:
# Hauptgruppe aus der Erzeugnisnummer ableiten (erste 6 Zeichen)
fassungen_mit_gruppen = fassungen.assign(
    hauptgruppe=fassungen["erzeugnis"].apply(L2.hauptgruppe_von)
)

# Jede Fassung muss einer Hauptgruppe zugeordnet sein
assert fassungen_mit_gruppen["hauptgruppe"].notna().all()

X_je_hauptgruppe = {}
info_je_hauptgruppe = {}

for hauptgruppe, gruppe in fassungen_mit_gruppen.groupby("hauptgruppe"):
    # Aus jedem Komponentenset eine Zeile je enthaltener Komponente machen
    vorhanden = (
        gruppe[["fassung_id", "komponentenset"]]
        .explode("komponentenset")
        .rename(columns={"komponentenset": "komponente"})
    )

    # Anwesenheit als 0/1-Tabelle darstellen
    X = pd.crosstab(vorhanden["fassung_id"], vorhanden["komponente"])
    X = X.reindex(index=gruppe["fassung_id"], fill_value=0)
    X = (X > 0).astype("int8")

    X_je_hauptgruppe[hauptgruppe] = X
    info_je_hauptgruppe[hauptgruppe] = (
        gruppe.set_index("fassung_id").loc[X.index].copy()
    )

display(pd.DataFrame([
    {
        "Hauptgruppe": hg,
        "Produkte": info_je_hauptgruppe[hg]["erzeugnis"].nunique(),
        "Fassungen": X.shape[0],
        "Verschiedene Komponenten": X.shape[1]
    }
    for hg, X in X_je_hauptgruppe.items()
]))

In [0]:
import numpy as np
import pandas as pd

HAUPTGRUPPE = "GJL121"
SEED = 42
N_FOLDS = 5

X_pilot = X_je_hauptgruppe[HAUPTGRUPPE].copy()
info_pilot = info_je_hauptgruppe[HAUPTGRUPPE].loc[X_pilot.index].copy()

# Produkte reproduzierbar mischen und auf fünf Prüfgruppen verteilen.
produkte = np.array(sorted(info_pilot["erzeugnis"].unique()))
np.random.default_rng(SEED).shuffle(produkte)

fold_je_produkt = {
    produkt: nummer % N_FOLDS
    for nummer, produkt in enumerate(produkte)
}
info_pilot["fold"] = info_pilot["erzeugnis"].map(fold_je_produkt)

# Alle Fassungen eines Produkts müssen im selben Fold liegen.
assert info_pilot.groupby("erzeugnis")["fold"].nunique().eq(1).all()
assert info_pilot.index.equals(X_pilot.index)

uebersicht = info_pilot.groupby("fold").agg(
    Pruefprodukte=("erzeugnis", "nunique"),
    Prueffassungen=("erzeugnis", "size"),
).reset_index()

uebersicht["Trainingsprodukte"] = len(produkte) - uebersicht["Pruefprodukte"]
display(uebersicht)

In [0]:
# BIG2 – Zelle 1: Tatsächliche Mengen- und Positionszustände vermessen

import numpy as np
import pandas as pd


big2_quelle = pd.DataFrame(sauber).copy()

pflichtfelder = [
    "erzeugnis",
    "block",
    "komponente",
    "menge",
    "tiefe",
    "alt_gruppe",
    "alt_wahrsch",
]

fehlende_felder = [
    feld for feld in pflichtfelder
    if feld not in big2_quelle.columns
]

assert not fehlende_felder, (
    f"Für Big2 fehlen diese Felder: {fehlende_felder}"
)


# Hauptgruppe und Tiefe sauber aufbereiten
big2_quelle["hauptgruppe"] = (
    big2_quelle["erzeugnis"]
    .apply(L2.hauptgruppe_von)
)

big2_quelle["tiefe_num"] = pd.to_numeric(
    big2_quelle["tiefe"],
    errors="coerce"
)


# Mengen robust in Zahlen umwandeln
def menge_als_zahl(wert):
    if pd.isna(wert):
        return np.nan

    if isinstance(wert, (int, float, np.integer, np.floating)):
        return float(wert)

    text = str(wert).strip().replace(" ", "")

    # Deutsches Dezimalkomma auffangen
    if "," in text and "." not in text:
        text = text.replace(",", ".")

    return pd.to_numeric(text, errors="coerce")


big2_quelle["menge_num"] = (
    big2_quelle["menge"]
    .map(menge_als_zahl)
)


# Alternativpositionen NICHT ausschließen
alt_text = (
    big2_quelle["alt_gruppe"]
    .fillna("")
    .astype(str)
    .str.strip()
)

big2_quelle["ist_alternative"] = (
    alt_text.ne("")
    & alt_text.ne("0")
    & alt_text.str.upper().ne("NAN")
)


# Konservativer erster Big2-Umfang:
# Hauptgruppe GJL121 und Tiefe 1.
# Menge 0 und Alternativpositionen bleiben ausdrücklich enthalten.
big2_positionen = big2_quelle.loc[
    big2_quelle["hauptgruppe"].eq(HAUPTGRUPPE)
    & big2_quelle["tiefe_num"].eq(1)
].copy()

assert not big2_positionen.empty
assert big2_positionen["erzeugnis"].isin(fold_je_produkt).all()

big2_positionen["fold"] = (
    big2_positionen["erzeugnis"]
    .map(fold_je_produkt)
)


# Mengen lesbar und stabil darstellen
def mengenwert_als_text(wert):
    if pd.isna(wert):
        return "FEHLT"

    return f"{float(wert):.12g}"


def mengenmuster(werte):
    werte = sorted(
        float(wert)
        for wert in werte
        if pd.notna(wert)
    )

    if not werte:
        return "FEHLT"

    return "|".join(
        mengenwert_als_text(wert)
        for wert in werte
    )


# Ein vorhandener Big2-Fall:
# Produkt + Block + exakte Komponente
big2_faelle = (
    big2_positionen
    .groupby(
        ["erzeugnis", "block", "komponente"],
        observed=True
    )
    .agg(
        positionsanzahl=("komponente", "size"),
        mengenmuster=("menge_num", mengenmuster),
        fehlende_mengenwerte=(
            "menge_num",
            lambda werte: int(werte.isna().sum())
        ),
        alternativpositionen=(
            "ist_alternative",
            "sum"
        ),
        fold=("fold", "first"),
    )
    .reset_index()
)


big2_faelle["zustand_vorhanden"] = (
    "POSITIONEN_"
    + big2_faelle["positionsanzahl"].astype(str)
    + "__MENGEN_"
    + big2_faelle["mengenmuster"]
)


# Vollständige Big2-Grundgesamtheit:
# Jeder Block × jede auf Tiefe 1 bekannte Komponente
bloecke = (
    big2_positionen[
        ["erzeugnis", "block", "fold"]
    ]
    .drop_duplicates()
)

komponenten = pd.DataFrame({
    "komponente": sorted(
        big2_positionen["komponente"].unique()
    )
})

big2_universum = bloecke.merge(
    komponenten,
    how="cross"
)

big2_universum = big2_universum.merge(
    big2_faelle[
        [
            "erzeugnis",
            "block",
            "komponente",
            "positionsanzahl",
            "mengenmuster",
            "fehlende_mengenwerte",
            "alternativpositionen",
            "zustand_vorhanden",
        ]
    ],
    on=["erzeugnis", "block", "komponente"],
    how="left",
    validate="one_to_one",
)

big2_universum["zustand_big2"] = (
    big2_universum["zustand_vorhanden"]
    .fillna("NICHT_GELISTET")
)

big2_universum["vorhanden_big2"] = (
    big2_universum["zustand_big2"]
    .ne("NICHT_GELISTET")
    .astype("int8")
)


# Mehrfachblöcke desselben Produkts untersuchen
blockanzahl = (
    bloecke.groupby("erzeugnis")["block"]
    .nunique()
)

mehrblock_produkte = blockanzahl[
    blockanzahl > 1
].index

blockabweichungen = (
    big2_universum.loc[
        big2_universum["erzeugnis"].isin(
            mehrblock_produkte
        )
    ]
    .groupby(
        ["erzeugnis", "komponente"]
    )["zustand_big2"]
    .nunique()
)

anzahl_blockabweichungen = int(
    blockabweichungen.gt(1).sum()
)


# Übersicht
display(pd.DataFrame([{
    "Produkte": big2_positionen["erzeugnis"].nunique(),
    "Produktblöcke": bloecke.shape[0],
    "Komponenten auf Tiefe 1":
        big2_positionen["komponente"].nunique(),
    "Positionszeilen": len(big2_positionen),
    "Big2-Fälle vorhanden": len(big2_faelle),
    "Big2-Paare vollständig": len(big2_universum),
    "Verschiedene Mengenwerte":
        big2_positionen["menge_num"].nunique(dropna=True),
    "Positionen mit Menge 0":
        int(big2_positionen["menge_num"].eq(0).sum()),
    "Positionen mit negativer Menge":
        int(big2_positionen["menge_num"].lt(0).sum()),
    "Positionen ohne lesbare Menge":
        int(big2_positionen["menge_num"].isna().sum()),
    "Alternativpositionen":
        int(big2_positionen["ist_alternative"].sum()),
    "Mehrblock-Produkte": len(mehrblock_produkte),
    "Produkt-Komponenten mit unterschiedlichen Blockzuständen":
        anzahl_blockabweichungen,
}]))


print("Häufigste einzelne Mengenwerte:")

display(
    big2_positionen
    .groupby("menge_num", dropna=False)
    .agg(
        Positionszeilen=("komponente", "size"),
        Produkte=("erzeugnis", "nunique"),
        Komponenten=("komponente", "nunique"),
    )
    .reset_index()
    .sort_values(
        "Positionszeilen",
        ascending=False
    )
    .head(30)
)


print("Häufigste vollständige Big2-Zustände:")

display(
    big2_universum
    .groupby("zustand_big2")
    .agg(
        Prüfpaare=("komponente", "size"),
        Produkte=("erzeugnis", "nunique"),
        Komponenten=("komponente", "nunique"),
    )
    .reset_index()
    .sort_values(
        "Prüfpaare",
        ascending=False
    )
    .head(30)
)


print("Alternativinformationen im Big2-Umfang:")

display(
    big2_positionen.assign(
        alt_wahrsch_anzeige=(
            big2_positionen["alt_wahrsch"]
            .fillna("OHNE_ANGABE")
            .astype(str)
        )
    )
    .groupby(
        ["ist_alternative", "alt_wahrsch_anzeige"],
        dropna=False
    )
    .agg(
        Positionszeilen=("komponente", "size"),
        Produkte=("erzeugnis", "nunique"),
    )
    .reset_index()
    .sort_values(
        "Positionszeilen",
        ascending=False
    )
)

In [0]:
# In Databricks als EINE Zelle nach der bisherigen Big2-Vorbereitung ausführen.
# Benötigt: big2_positionen (Tiefe 1, GJL121, Spalte menge_num)
# und fold_je_produkt aus MAIN. Ersetzt die bisherige Big2-Trainingszelle.

import time
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier

PRUEF_FOLD_BIG2 = 0  # Erst ein produktweise zurückgehaltener Fold.

positionen_b2 = pd.DataFrame(big2_positionen).copy()
pflicht = {"erzeugnis", "block", "komponente", "menge_num"}
fehlende_spalten = pflicht - set(positionen_b2.columns)
assert not fehlende_spalten, f"Big2-Eingabe ohne Spalten: {fehlende_spalten}"
assert positionen_b2[list(pflicht)].notna().all().all(), (
    "Produkt, Block, Komponente oder Menge fehlt: zuerst die Vorbereitung prüfen."
)
positionen_b2["erzeugnis"] = positionen_b2["erzeugnis"].astype(str)
positionen_b2["komponente"] = positionen_b2["komponente"].astype(str)
positionen_b2["menge_num"] = pd.to_numeric(
    positionen_b2["menge_num"], errors="raise"
).astype(float)
assert np.isfinite(positionen_b2["menge_num"]).all()

# Mengen/Positionen derselben Materialnummer in einem Produktblock bleiben zusammen.
faelle_b2 = (
    positionen_b2.groupby(["erzeugnis", "block", "komponente"], sort=False)["menge_num"]
    .agg(lambda werte: tuple(sorted(werte)))
    .reset_index(name="mengen_je_position")
)
faelle_b2["anzahl_positionen"] = faelle_b2["mengen_je_position"].map(len)
faelle_b2["menge_summe"] = faelle_b2["mengen_je_position"].map(sum)
faelle_b2["zustand_big2"] = faelle_b2["mengen_je_position"].map(
    lambda werte: "POSITIONEN_" + str(len(werte)) + "__MENGEN_"
    + "|".join(format(float(wert), ".12g") for wert in werte)
)
assert faelle_b2["anzahl_positionen"].sum() == len(positionen_b2)

# Im vorliegenden Pilot hatten alle mehrfachen Positionen dieselbe Einzelmenge.
# Bei gemischten Mengen reichen Anzahl + Summe im KONTEXT nicht zur Rekonstruktion.
# Lieber sichtbar stoppen als die Information unbemerkt wegwerfen.
gemischte_mengen = faelle_b2["mengen_je_position"].map(
    lambda werte: len(set(werte)) > 1
)
assert not gemischte_mengen.any(), (
    "Gemischte Einzelmengen in einem Produktblock/Material: "
    "Kontextdarstellung um Einzelmengen erweitern."
)

block_index = pd.MultiIndex.from_frame(
    faelle_b2[["erzeugnis", "block"]].drop_duplicates()
)
komponenten_b2 = sorted(faelle_b2["komponente"].unique())

positionen_matrix = (
    faelle_b2.pivot(index=["erzeugnis", "block"], columns="komponente",
                    values="anzahl_positionen")
    .reindex(index=block_index, columns=komponenten_b2)
    .fillna(0).to_numpy(dtype=np.float32)
)
mengen_matrix = (
    faelle_b2.pivot(index=["erzeugnis", "block"], columns="komponente",
                    values="menge_summe")
    .reindex(index=block_index, columns=komponenten_b2)
    .fillna(0).to_numpy(dtype=np.float32)
)

block_je_fall = block_index.get_indexer(
    pd.MultiIndex.from_frame(faelle_b2[["erzeugnis", "block"]])
)
spalte_je_fall = pd.Index(komponenten_b2).get_indexer(faelle_b2["komponente"])
assert (block_je_fall >= 0).all() and (spalte_je_fall >= 0).all()
zeilen_b2 = np.arange(len(faelle_b2))

# Gleicher Mechanismus wie Big1: eigene Antwort verdecken, andere Komponenten sehen.
kontext_positionen = positionen_matrix[block_je_fall].copy()
kontext_mengen = mengen_matrix[block_je_fall].copy()
assert np.array_equal(
    kontext_positionen[zeilen_b2, spalte_je_fall],
    faelle_b2["anzahl_positionen"].to_numpy(dtype=np.float32)
)
assert np.allclose(
    kontext_mengen[zeilen_b2, spalte_je_fall],
    faelle_b2["menge_summe"].to_numpy(dtype=np.float32)
)
kontext_positionen[zeilen_b2, spalte_je_fall] = 0
kontext_mengen[zeilen_b2, spalte_je_fall] = 0

features_big2_neu = pd.DataFrame(
    np.concatenate([kontext_positionen, kontext_mengen], axis=1),
    columns=([f"anzahl_kontext_{i}" for i in range(len(komponenten_b2))]
             + [f"menge_kontext_{i}" for i in range(len(komponenten_b2))]),
)
features_big2_neu["pruefkomponente"] = faelle_b2["komponente"].to_numpy()
assert (kontext_positionen[zeilen_b2, spalte_je_fall] == 0).all()
assert (kontext_mengen[zeilen_b2, spalte_je_fall] == 0).all()

fold_map_b2 = {str(produkt): fold for produkt, fold in fold_je_produkt.items()}
faelle_b2["fold"] = faelle_b2["erzeugnis"].map(fold_map_b2)
assert faelle_b2["fold"].notna().all(), "Big2-Produkt ohne Big1-Fold."
assert faelle_b2.groupby("erzeugnis")["fold"].nunique().eq(1).all()

blockanzahl_b2 = (
    faelle_b2[["erzeugnis", "block"]].drop_duplicates()
    .groupby("erzeugnis").size()
)
faelle_b2["blockgewicht"] = 1.0 / faelle_b2["erzeugnis"].map(blockanzahl_b2)

train_b2 = faelle_b2["fold"].ne(PRUEF_FOLD_BIG2).to_numpy()
pruefen_b2 = ~train_b2
assert train_b2.any() and pruefen_b2.any()
assert set(faelle_b2.loc[train_b2, "erzeugnis"]).isdisjoint(
    faelle_b2.loc[pruefen_b2, "erzeugnis"]
)

print(
    f"Big2: {len(faelle_b2)} vorhandene Produktblock-Komponenten-Fälle, "
    f"{int(train_b2.sum())} Training, {int(pruefen_b2.sum())} Prüfung, "
    f"{faelle_b2.loc[train_b2, 'zustand_big2'].nunique()} "
    f"Trainingszustände. Starte EIN Modell.",
    flush=True,
)
start_b2 = time.perf_counter()
modell_big2_pilot = CatBoostClassifier(
    loss_function="MultiClass",
    iterations=300,
    depth=5,
    learning_rate=0.06,
    random_seed=42 + PRUEF_FOLD_BIG2,
    thread_count=8,
    verbose=10,
    allow_writing_files=False,
)
modell_big2_pilot.fit(
    features_big2_neu.loc[train_b2],
    faelle_b2.loc[train_b2, "zustand_big2"],
    cat_features=["pruefkomponente"],
    sample_weight=faelle_b2.loc[train_b2, "blockgewicht"],
)
print(f"Big2-Training fertig nach {time.perf_counter() - start_b2:.1f} s.", flush=True)

# Rückgehaltene Produkte bewerten. Ein im Training völlig unbekannter Zustand
# bekommt KEINE erfundene Wahrscheinlichkeit von 0 und keinen Score von 1.
wahrscheinlichkeiten_b2 = modell_big2_pilot.predict_proba(
    features_big2_neu.loc[pruefen_b2]
)
klassen_b2 = np.asarray(modell_big2_pilot.classes_).astype(str)
klassen_index_b2 = {klasse: i for i, klasse in enumerate(klassen_b2)}
ergebnisse_big2_pilot = faelle_b2.loc[pruefen_b2].copy().reset_index(drop=True)
ergebnisse_big2_pilot["erwarteter_zustand"] = klassen_b2[
    wahrscheinlichkeiten_b2.argmax(axis=1)
]
ergebnisse_big2_pilot["p_tatsaechlicher_zustand"] = [
    float(wahrscheinlichkeiten_b2[i, klassen_index_b2[zustand]])
    if zustand in klassen_index_b2 else np.nan
    for i, zustand in enumerate(ergebnisse_big2_pilot["zustand_big2"])
]
ergebnisse_big2_pilot["auffaelligkeit_big2"] = (
    1 - ergebnisse_big2_pilot["p_tatsaechlicher_zustand"]
)
ergebnisse_big2_pilot["pruefstatus_big2"] = np.where(
    ergebnisse_big2_pilot["p_tatsaechlicher_zustand"].notna(),
    "bewertbar", "zustand_nicht_im_training",
)

# Die interne Zielklasse enthält Einzelpositionen; die Anzeige zeigt nur Mengen.
train_zustaende_b2 = (
    faelle_b2.loc[train_b2]
    .drop_duplicates("zustand_big2")
    .set_index("zustand_big2")
)
ergebnisse_big2_pilot["beobachtete_menge"] = (
    ergebnisse_big2_pilot["menge_summe"]
)
ergebnisse_big2_pilot["erwartete_menge"] = (
    ergebnisse_big2_pilot["erwarteter_zustand"]
    .map(train_zustaende_b2["menge_summe"])
)
erwartete_anzahl_b2 = (
    ergebnisse_big2_pilot["erwarteter_zustand"]
    .map(train_zustaende_b2["anzahl_positionen"])
)
gleiche_menge_b2 = np.isclose(
    ergebnisse_big2_pilot["beobachtete_menge"],
    ergebnisse_big2_pilot["erwartete_menge"],
)
ergebnisse_big2_pilot["grund_big2"] = np.select(
    [
        ~gleiche_menge_b2,
        ergebnisse_big2_pilot["anzahl_positionen"] > erwartete_anzahl_b2,
        ergebnisse_big2_pilot["anzahl_positionen"] < erwartete_anzahl_b2,
    ],
    ["Menge abweichend", "möglicherweise doppelte Position",
     "Verteilung auffällig"],
    default="Mengenmuster auffällig",
)

belege_b2 = faelle_b2.loc[train_b2].groupby("komponente")["erzeugnis"].nunique()
ergebnisse_big2_pilot["trainingsprodukte_komponente"] = (
    ergebnisse_big2_pilot["komponente"].map(belege_b2).fillna(0).astype(int)
)
belege_zustand_b2 = (
    faelle_b2.loc[train_b2]
    .groupby(["komponente", "zustand_big2"])["erzeugnis"].nunique()
)
ergebnisse_big2_pilot["trainingsprodukte_dieser_zustand"] = [
    int(belege_zustand_b2.get((k, z), 0))
    for k, z in zip(ergebnisse_big2_pilot["komponente"],
                    ergebnisse_big2_pilot["zustand_big2"])
]

# Materialnummer bleibt die Identität; Namen dienen nur der Anzeige.
namen_b2 = pd.DataFrame(sauber).copy()
if {"erzeugnis", "erzeugnis_txt", "komponente", "komponente_txt"}.issubset(namen_b2.columns):
    namen_b2["erzeugnis"] = namen_b2["erzeugnis"].astype(str)
    namen_b2["komponente"] = namen_b2["komponente"].astype(str)
    produktnamen_b2 = (
        namen_b2.drop_duplicates("erzeugnis").set_index("erzeugnis")["erzeugnis_txt"]
    )
    komponentennamen_b2 = (
        namen_b2.drop_duplicates("komponente").set_index("komponente")["komponente_txt"]
    )
    ergebnisse_big2_pilot["erzeugnis_txt"] = (
        ergebnisse_big2_pilot["erzeugnis"].map(produktnamen_b2)
    )
    ergebnisse_big2_pilot["komponente_txt"] = (
        ergebnisse_big2_pilot["komponente"].map(komponentennamen_b2)
    )

display(pd.DataFrame([{
    "Trainierte Modelle": 1,
    "Trainingsfälle": int(train_b2.sum()),
    "Prüffälle": int(pruefen_b2.sum()),
    "Bewertbare Prüffälle": int(ergebnisse_big2_pilot["auffaelligkeit_big2"].notna().sum()),
    "Neue Zustände ohne Score": int(ergebnisse_big2_pilot["auffaelligkeit_big2"].isna().sum()),
}]))


print("Nicht gelernte Zustände separat (ohne künstlichen 1,0-Score):")
display(
    ergebnisse_big2_pilot.loc[
        ergebnisse_big2_pilot["auffaelligkeit_big2"].isna(), ausgabe_b2
    ].head(20)
    .rename(columns={"erzeugnis": "Produktnummer",
                     "erzeugnis_txt": "Produktname",
                     "komponente": "Komponentennummer",
                     "komponente_txt": "Komponentenname",
                     "beobachtete_menge": "Menge aktuell",
                     "erwartete_menge": "Menge erwartet",
                     "grund_big2": "Grund",
                     "auffaelligkeit_big2": "Auffälligkeit Big2"})
)


In [0]:
ausgabe_b2 = [
    #"erzeugnis", "erzeugnis_txt", "komponente", "komponente_txt",
    "beobachtete_menge", "erwartete_menge", "grund_big2", "auffaelligkeit_big2",
]
ausgabe_b2 = [s for s in ausgabe_b2 if s in ergebnisse_big2_pilot.columns]
display(
    ergebnisse_big2_pilot.loc[
        ergebnisse_big2_pilot["auffaelligkeit_big2"].notna(), ausgabe_b2
    ].sort_values("auffaelligkeit_big2", ascending=False).head(100)
    .rename(columns={#"erzeugnis": "Produktnummer",
                     #"erzeugnis_txt": "Produktname",
                     #"komponente": "Komponentennummer",
                     #"komponente_txt": "Komponentenname",
                     "beobachtete_menge": "Menge aktuell",
                     "erwartete_menge": "Menge erwartet",
                     "grund_big2": "Grund",
                     "auffaelligkeit_big2": "Auffälligkeit Big2"})
)